In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class SelfAttention(nn.Module):
  def __init__(self, model_dim, num_heads=6):
    super().__init__()
    self.WQ = nn.Linear(model_dim, model_dim) # num_heads * head_dim = model_dim
    self.WK = nn.Linear(model_dim, model_dim)
    self.WV = nn.Linear(model_dim, model_dim)
    self.WO = nn.Linear(model_dim, model_dim)
    self.model_dim = model_dim
    self.num_heads = num_heads
    self.head_dim = model_dim // num_heads

    assert model_dim % num_heads == 0

  def forward(self, q, k, v):
    B, L, _ = q.shape

    Q = self.WQ(q) # [B, Seq_len, model_dim]
    Q = torch.reshape(Q, (B, L, self.num_heads, self.head_dim)) # [B, Seq_len, num_heads, heads_dim]
    Q = torch.permute(Q, (0, 2, 1, 3)) # [B, num_heads, seq_len, heads_dim]

    K = self.WK(k)
    K = torch.reshape(K, (B, L, self.num_heads, self.head_dim))
    K = torch.permute(K, (0, 2, 1, 3))

    V = self.WV(v)
    V = torch.reshape(V, (B, L, self.num_heads, self.head_dim))
    V = torch.permute(V, (0, 2, 1, 3))

    scores = torch.matmul(Q, K.transpose(-2, -1)) # [B, num_heads, L, L]

    scores = scores / math.sqrt(self.head_dim)

    mask = torch.triu(torch.ones(L, L), diagonal=1).bool()

    scores = scores.masked_fill(mask, float('-inf'))

    out = torch.softmax(scores, dim=-1)

    final = torch.matmul(out, V) # [B, num_heads, seq_len, heads_dim]

    final = torch.permute(final, (0, 2, 1, 3)) # [B, seq_len, num_heads, heads_dim]

    final = torch.reshape(final, (B, L, self.model_dim)) #[B, seq_len, model_dim]

    return self.WO(final)

#att = MultiHeadAttention(516)

In [ ]:
class DecoderBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.5):
    super().__init__()
    self.attention = SelfAttention(d_model, num_heads)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

    self.FFN = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    # in -> [B, L, d_model]
    # out -> [B, L, d_model]

    norm_x = self.ln1(x)
    atrakcja = self.attention(norm_x, norm_x, norm_x) # [B, L, d_model]
    x = x + self.dropout(atrakcja)

    norm_x2 = self.ln2(x)
    ffn_out = self.FFN(norm_x2)
    x = x + self.dropout(ffn_out)

    return x


In [ ]:
def get_sinusoidal_embeddings(seq_len, d_model):
  # out -> [seq_len, d_model]

  positions = torch.arange(0, seq_len).unsqueeze(1) # [seq_len]

  dims = torch.arange(0, d_model, 2).unsqueeze(0) # [d_model//2]

  mian = torch.exp(torch.log(torch.tensor(10000, dtype=torch.float32))*(dims/d_model))

  angles = positions/mian

  #print(final.shape) [50, 128]
  w, _ = angles.shape

  #print(w)
  #print(h)

  final = torch.ones(w, d_model)

  final[:, 0::2] = torch.sin(angles)
  final[:, 1::2] = torch.cos(angles)

  return final

#print(get_sinusoidal_embeddings(50, 256))

In [ ]:
class DecoderTransformer(nn.Module):
  def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
    super().__init__()
    self.emb = nn.Embedding(vocab_size, d_model)
    self.model_dim = d_model
    self.lin = nn.Linear(d_model, vocab_size)
    self.blocks = nn.ModuleList()

    for i in range(num_layers):
      self.blocks.append(DecoderBlock(d_model, num_heads, d_ff))

  def forward(self, x):
    # in -> [B, L]
    # out -> [B, L, vocab_size]

    B, L = x.shape

    x = self.emb(x) + get_sinusoidal_embeddings(L, self.model_dim) # [B, L, d_model]

    for block in self.blocks:
      x = block(x)

    #print(x.shape)
    x = self.lin(x)
    #print(x.shape)

    return x

